# Veri Hazırlama

**Proje:** ABD Trafik Kazalarında Hava Durumu, Yol Tipi ve Saat Bilgisine Göre
Kaza Şiddetinin Tahmini

**Kapsam:** Veri hacmi nedeniyle analiz Florida, New York ve Minnesota
eyaletleriyle sınırlandırılmıştır. Bu üç eyalet farklı iklim koşullarını
temsil edecek şekilde seçilmiştir.

Bu notebook ham US Accidents verisini okur, temizler, dış kaynaklı ilçe
verisiyle zenginleştirir ve modellemeye hazır hale getirip kaydeder.

**Girdi:** `US_Accidents_March23.csv`, `county_referans_kendi.csv`  
**Çıktı:** `temiz_veri.parquet`

In [1]:
#Uzun süren işlemler öncesinde girdi dosyalarının varlığını doğrulama
import pandas as pd
import numpy as np
import re
import os

RAW  = "../data/raw/"
PROC = "../data/processed/"

ANA_VERI = RAW + "US_Accidents_March23.csv"
REFERANS = RAW + "county_referans_kendi.csv"

EYALETLER = ["FL", "NY", "MN"]

for yol in [ANA_VERI, REFERANS]:#yol bir geçici değişken. Her turda listeden bir sonraki öğeyi alıyor:
    if os.path.exists(yol):#diske bakar, dosya varsa True döner.
        mb = os.path.getsize(yol) / 1024**2#	Boyut (bayt)
        print(f"✓ {os.path.basename(yol):<32} {mb:>8.1f} MB")
    else:
        print(f"✗ BULUNAMADI: {yol}")

os.makedirs(PROC, exist_ok=True)#processed klasörünü oluşturuyor.

✓ US_Accidents_March23.csv           2916.5 MB
✓ county_referans_kendi.csv             0.2 MB


In [2]:
ATILACAK = ["End_Lat", "End_Lng", "Wind_Chill(F)", "Distance(mi)",
            "End_Time", "Description", "Country", "Source",
            "Turning_Loop", "Weather_Timestamp", "Airport_Code"]

parcalar = []
okuyucu = pd.read_csv(ANA_VERI, chunksize=500_000, low_memory=False)

for i, parca in enumerate(okuyucu, 1):
    secili = parca[parca["State"].isin(EYALETLER)]
    parcalar.append(secili)
    print(f"{i:>2}. parça — {len(secili):>7,} satır alındı")

df = pd.concat(parcalar, ignore_index=True)
df = df.drop(columns=ATILACAK)

del parcalar #Döngü bitti ama parcalar listesi hâlâ bellekte,del onu siliyor, yaklaşık 1 GB boşalıyor.

print()
print("Toplam satır:", f"{len(df):,}")
print("Sütun sayısı:", df.shape[1])
print()
print(df["State"].value_counts())

 1. parça —  74,548 satır alındı
 2. parça —  70,126 satır alındı
 3. parça —  75,798 satır alındı
 4. parça —  76,230 satır alındı
 5. parça —  75,437 satır alındı
 6. parça —  68,603 satır alındı
 7. parça —  76,381 satır alındı
 8. parça —  94,775 satır alındı
 9. parça — 115,995 satır alındı
10. parça — 115,243 satır alındı
11. parça — 116,719 satır alındı
12. parça — 122,991 satır alındı
13. parça — 123,131 satır alındı
14. parça — 121,793 satır alındı
15. parça —  59,776 satır alındı
16. parça —  32,690 satır alındı

Toplam satır: 1,420,236
Sütun sayısı: 35

State
FL    880192
NY    347960
MN    192084
Name: count, dtype: int64


## 2. Eksik değerlerin doldurulması

Eksik değerler sütun türüne göre farklı stratejilerle ele alınmıştır:
yağış sütununda eksiklik "yağış yok" anlamına geldiği için sıfır,
sayısal sütunlarda aykırı değerlerden etkilenmemek için medyan,
kategorik sütunlarda ise ayrı bir sınıf olarak "Bilinmiyor" kullanılmıştır.

In [3]:
# --- ÖNCE ---
eksik = pd.DataFrame({
    "eksik_sayi": df.isna().sum(),
    "eksik_yuzde": (df.isna().mean() * 100).round(2)
})
eksik = eksik[eksik.eksik_sayi > 0].sort_values("eksik_yuzde", ascending=False)

print("DOLDURMADAN ÖNCE")
print(eksik.to_string())
print()

# --- 1. Yağış: eksik = yağış yoktu ---Boşluk aslında "yağmur yoktu" demek. Gerçekten 0 mm yağış vardı
df["Precipitation(in)"] = df["Precipitation(in)"].fillna(0)

# --- 2. Sayısal: medyan ---. Ortalama yerine onu seçtin çünkü ortalama uç değerlerden etkileniyor
SAYISAL = ["Temperature(F)", "Humidity(%)", "Pressure(in)",
           "Visibility(mi)", "Wind_Speed(mph)"]
for s in SAYISAL:
    df[s] = df[s].fillna(df[s].median())

# --- 3. Kategorik: "Bilinmiyor" ---
METIN = ["Weather_Condition", "Wind_Direction", "City", "Zipcode",
         "Street", "Timezone", "Sunrise_Sunset", "Civil_Twilight",
         "Nautical_Twilight", "Astronomical_Twilight"]
for m in METIN:
    df[m] = df[m].fillna("Bilinmiyor")

# --- SONRA ---
kalan = df.isna().sum()
print("DOLDURMADAN SONRA")
print("Toplam eksik:", kalan.sum())
if kalan.sum() > 0:
    print(kalan[kalan > 0].to_string())

DOLDURMADAN ÖNCE
                       eksik_sayi  eksik_yuzde
Precipitation(in)          289827        20.41
Wind_Speed(mph)             60145         4.23
Wind_Direction              21934         1.54
Humidity(%)                 20581         1.45
Temperature(F)              18623         1.31
Visibility(mi)              17632         1.24
Weather_Condition           15700         1.11
Pressure(in)                12907         0.91
Sunrise_Sunset               5395         0.38
Civil_Twilight               5395         0.38
Nautical_Twilight            5395         0.38
Astronomical_Twilight        5395         0.38
Street                       1899         0.13
Zipcode                       800         0.06
Timezone                      801         0.06
City                           66         0.00

DOLDURMADAN SONRA
Toplam eksik: 0


## 3. Dış veri kaynağının birleştirilmesi

US Census Bureau ve CDC/NCHS verilerinden türetilen ilçe düzeyindeki
nüfus, nüfus yoğunluğu ve kentsel-kırsal sınıflandırma bilgileri,
State + ilçe adı anahtarı üzerinden kaza verisine eklenmiştir.

In [4]:
# Referans tablosunu oku
ref = pd.read_csv(REFERANS)
print("Referans tablosu:", ref.shape)
print(ref.columns.tolist())
print()

# Kaza tarafında eşleşme anahtarını üret
df["county_key"] = (df["County"]
                    .str.lower()
                    .str.replace(".", "", regex=False)
                    .str.strip())

# Birleştir
oncesi = len(df)
df = df.merge(
    ref[["State", "county_key", "nufus_2022",
         "nufus_yogunlugu", "kentsel_kirsal"]],
    on=["State", "county_key"],#Sadece ilçe adıyla eşleştirseydik felaket olurdu. ABD'de 31 tane "Washington County" var. Farklı eyaletlerde, farklı yerler.
    how="left"
)

print("Satır sayısı:", oncesi, "→", len(df))
print("Eşleşmeyen oran:", f"{df['nufus_2022'].isna().mean():.2%}")
print()
print(df[["State", "County", "nufus_2022",
          "nufus_yogunlugu", "kentsel_kirsal"]].head(5).to_string())

Referans tablosu: (3138, 10)
['State', 'county_key', 'GEOID', 'NAME', 'STNAME', 'nufus_2022', 'alan_sqmi', 'nufus_yogunlugu', 'kentsel_kirsal', 'birlesik_kayit']

Satır sayısı: 1420236 → 1420236
Eşleşmeyen oran: 0.01%

  State        County  nufus_2022  nufus_yogunlugu  kentsel_kirsal
0    FL  Hillsborough   1523839.0          1490.25             1.0
1    FL  Hillsborough   1523839.0          1490.25             1.0
2    FL    Miami-Dade   2713415.0          1428.15             1.0
3    FL    Miami-Dade   2713415.0          1428.15             1.0
4    FL       Broward   1966237.0          1634.58             2.0


## 4. Öznitelik mühendisliği

Modelin kullanabileceği yeni değişkenler türetilmiştir. Start_Time
sütunundan zaman bileşenleri (saat, gün, mevsim vb.), Street sütunundan
ise düzenli ifadelerle yol tipi kategorisi çıkarılmıştır. Bu değişkenler
dış kaynaktan gelmemekte, mevcut veriden üretilmektedir.

### 4.1 Zaman değişkenleri

Start_Time sütunu metin formatındadır ve model tarafından doğrudan
kullanılamaz. Tarih tipine dönüştürülerek saat, gün, ay, yıl bileşenleri
çıkarılmış; bunlardan hafta sonu, yoğun saat ve mevsim değişkenleri
türetilmiştir.

In [5]:
df["Start_Time"] = pd.to_datetime(df["Start_Time"], format="mixed")

df["saat"]       = df["Start_Time"].dt.hour
df["gun"]        = df["Start_Time"].dt.dayofweek     # 0=Pazartesi
df["ay"]         = df["Start_Time"].dt.month
df["yil"]        = df["Start_Time"].dt.year
df["hafta_sonu"] = df["gun"] >= 5
df["yogun_saat"] = df["saat"].isin([7, 8, 9, 16, 17, 18])

def mevsim(ay):
    if ay in [12, 1, 2]:  return "Kis"
    if ay in [3, 4, 5]:   return "Ilkbahar"
    if ay in [6, 7, 8]:   return "Yaz"
    return "Sonbahar"

df["mevsim"] = df["ay"].apply(mevsim)

print("Boyut:", df.shape)
print()
print("Mevsim dağılımı:")
print(df["mevsim"].value_counts())
print()
print("Örnek dönüşüm:")
print(df[["Start_Time", "saat", "gun", "hafta_sonu",
          "yogun_saat", "mevsim"]].head(3).to_string())

Boyut: (1420236, 46)

Mevsim dağılımı:
mevsim
Kis         435384
Sonbahar    383558
Ilkbahar    308467
Yaz         292827
Name: count, dtype: int64

Örnek dönüşüm:
           Start_Time  saat  gun  hafta_sonu  yogun_saat    mevsim
0 2016-11-30 15:36:03    15    2       False       False  Sonbahar
1 2016-11-30 16:25:35    16    2       False        True  Sonbahar
2 2016-11-30 16:40:31    16    2       False        True  Sonbahar


### 4.1 Yol tipi — ilk deneme

Sokak isimlerinden yol tipi çıkarmak için düzenli ifadeler kullanılmıştır.
İlk kural setinde yalnızca numaralı yollar (I-75, US-35) ve yaygın şehir içi
sokak ekleri (Ave, Blvd, St) hedeflenmiştir.

In [6]:
def yol_tipi_v1(s):
    s = str(s)
    if re.search(r"\bI-\d+", s):                              return "Otoyol"
    if re.search(r"\bUS-\d+|US Highway", s):                  return "Federal yol"
    if re.search(r"State (Route|Rte)|\b[A-Z]{2}-\d+", s):     return "Eyalet yolu"
    if re.search(r"County (Hwy|Road)|\bCR-\d+", s):           return "Ilce yolu"
    if re.search(r"\b(Dr|Ave|St|Ln|Ct|Blvd|Rd|Way|Pl)\b", s): return "Sehir ici"
    return "Diger"

v1 = df["Street"].apply(yol_tipi_v1)

print("İLK DENEME")
print(v1.value_counts())
print()
print("Sınıflandırılamayan oran:", f"{(v1 == 'Diger').mean():.1%}")
print()
print("En sık sınıflandırılamayan sokaklar:")
print(df.loc[v1 == "Diger", "Street"].value_counts().head(20).to_string())

İLK DENEME
Street
Sehir ici      638201
Diger          386594
Otoyol         298637
Federal yol     45889
Eyalet yolu     44900
Ilce yolu        6015
Name: count, dtype: int64

Sınıflandırılamayan oran: 27.2%

En sık sınıflandırılamayan sokaklar:
Street
Florida's Tpke S          9647
Brooklyn Queens Expy      9549
Florida's Tpke N          9065
Palmetto Expy S           7508
Long Island Expy W        7501
Long Island Expy E        7261
 S Orange Blossom Trl     6798
 S Dixie Hwy              6351
Palmetto Expy N           5911
Florida's Tpke            4448
Southern State Pkwy E     3937
Major Deegan Expy N       3386
Southern State Pkwy W     3365
Ronald Reagan Tpke        3360
W Beltway S               3232
Dolphin Expy E            3153
 S Tamiami Trl            3062
Adirondack Northway N     2845
E Beltway N               2829
New York State Thruway    2755


### 4.2 Yol tipi — genişletilmiş kural seti

İlk denemede sınıflandırılamayan kayıtlar incelendiğinde, bunların büyük
çoğunluğunun Turnpike, Expressway, Parkway ve Thruway gibi isimli hızlı
yollar olduğu görülmüştür. Bu yollar numaralı bir kalıp içermedikleri için
ilk kural setine takılmamıştır. Kurallar bu desenleri kapsayacak biçimde
genişletilmiştir.

In [7]:
def yol_tipi(s):
    s = str(s)
    # Numaralı eyaletler arası otoyol
    if re.search(r"\bI-\d+", s):
        return "Otoyol"
    # İsimli hızlı yollar — ilk denemede kaçanlar
    if re.search(r"\b(Expy|Expressway|Tpke|Turnpike|Pkwy|Parkway|Fwy|Freeway|Thruway|Beltway|Northway|Skyway|Causeway)\b", s, re.I):#re.I eklendi. "Ignore case" — büyük/küçük harf farkını yok sayar.
        return "Otoyol"
    if re.search(r"\bUS-\d+|US Highway", s):
        return "Federal yol"
    if re.search(r"State (Route|Rte)|\b[A-Z]{2}-\d+", s):
        return "Eyalet yolu"
    if re.search(r"County (Hwy|Road)|\bCR-\d+", s):
        return "Ilce yolu"
    if re.search(r"\b(Hwy|Highway)\b", s, re.I):
        return "Federal yol"
    if re.search(r"\b(Dr|Drive|Ave|Avenue|St|Street|Ln|Lane|Ct|Court|Blvd|Boulevard|Rd|Road|Way|Pl|Place|Pike|Trl|Trail|Cir|Circle|Ter|Terrace|Loop|Row|Aly|Alley|Sq|Square|Plz|Plaza|Bridge|Tunnel)\b", s, re.I):
        return "Sehir ici"
    return "Diger"

df["yol_tipi"] = df["Street"].apply(yol_tipi)

print("GENİŞLETİLMİŞ KURAL SETİ")
print(df["yol_tipi"].value_counts())
print()
print("Sınıflandırılamayan oran:", f"{(df['yol_tipi']=='Diger').mean():.1%}")
print("(ilk denemede %27.2 idi)")
print()
print("Hâlâ sınıflandırılamayanlar:")
print(df.loc[df["yol_tipi"]=="Diger", "Street"].value_counts().head(10).to_string())

GENİŞLETİLMİŞ KURAL SETİ
yol_tipi
Sehir ici      690357
Otoyol         518920
Federal yol    116779
Eyalet yolu     44900
Diger           43265
Ilce yolu        6015
Name: count, dtype: int64

Sınıflandırılamayan oran: 3.0%
(ilk denemede %27.2 idi)

Hâlâ sınıflandırılamayanlar:
Street
New England Trwy S             2461
Bilinmiyor                     1899
New York Trwy W                1772
George Washington Brg          1733
Central Florida Greeneway S    1330
New York Trwy N                1296
 Route 9                        834
Shakopee Byp N                  790
New England Trwy N              771
Central Florida Greeneway N     768


## 5. İşlenmiş verinin kaydedilmesi

Temizlenmiş ve zenginleştirilmiş veri, sütun tabanlı Parquet formatında
saklanmıştır. Bu format veri tiplerini koruduğu ve okuma süresini
saniyeler seviyesine indirdiği için sonraki notebook'larda ham veriye
tekrar dönülmesi gerekmemektedir.

In [8]:
CIKTI = PROC + "temiz_veri.parquet"

df.to_parquet(CIKTI, index=False)

boyut = os.path.getsize(CIKTI) / 1024**2
print("Kaydedildi:", CIKTI)
print(f"Dosya boyutu: {boyut:.1f} MB")
print(f"Satır: {len(df):,}   Sütun: {df.shape[1]}")
print()
print("Sütunlar:")
print(df.columns.tolist())

Kaydedildi: ../data/processed/temiz_veri.parquet
Dosya boyutu: 62.4 MB
Satır: 1,420,236   Sütun: 47

Sütunlar:
['ID', 'Severity', 'Start_Time', 'Start_Lat', 'Start_Lng', 'Street', 'City', 'County', 'State', 'Zipcode', 'Timezone', 'Temperature(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Direction', 'Wind_Speed(mph)', 'Precipitation(in)', 'Weather_Condition', 'Amenity', 'Bump', 'Crossing', 'Give_Way', 'Junction', 'No_Exit', 'Railway', 'Roundabout', 'Station', 'Stop', 'Traffic_Calming', 'Traffic_Signal', 'Sunrise_Sunset', 'Civil_Twilight', 'Nautical_Twilight', 'Astronomical_Twilight', 'county_key', 'nufus_2022', 'nufus_yogunlugu', 'kentsel_kirsal', 'saat', 'gun', 'ay', 'yil', 'hafta_sonu', 'yogun_saat', 'mevsim', 'yol_tipi']
